# പാഠം 03 - ഏജന്റിക് ഡിസൈൻ മാതൃകകൾ

ഈ പഠനത്തിൽ, ഫലപ്രദമായ AI ഏജന്റുകൾ നിർമ്മിക്കാൻ മൂന്ന് അടിസ്ഥാനപരമായ ഡിസൈൻ മാതൃകകൾ പരിശോധിക്കുന്നു:

1. **স্পষ্ট ഏജന്റ് നിർദ്ദേശങ്ങൾ** — ഏജന്റിന്റെ പ്രവർത്തനം മാർഗനിർദേശിക്കുന്ന കൃത്യമായ, വേഷം നിർവചിക്കുന്ന പ്രോംപ്റ്റുകൾ രൂപകൽപ്പന ചെയ്യുക
2. **Pydantic മോഡലുകളോടെ ഘടനാപരമായ ഔട്ട്പുട്ട്** — ഏജന്റുകൾ പ്രവചിക്കാവുന്ന, സാധുവായ ഡാറ്റ തിരികെ നൽകുന്നത് ഉറപ്പാക്കുന്നതിന്
3. **ഏകപരാധി ഏജന്റുകൾ** — ഓരോന്നും സുതാര്യമായി ഒരു കാര്യത്തിൽ നിഷ്പക്ഷമായി പ്രവർത്തിക്കുന്ന ഏജന്റുകളുടെ രൂപകൽപ്പന

ഞങ്ങൾ ഓരോ മാതൃകയും **പ്രവാസ ഗമ്യസ്ഥലം ശുപാർശക്കർത്താവ്** സംഭവവികാസത്തിൽ പ്രയോഗിച്ച്, നിർദേശം നൽകാനും, ലഭ്യത പരിശോധിക്കാനും, ലജിസ്റ്റിക്‌സ് കൈകാര്യം ചെയ്യാനും കഴിയുന്ന ഒരു സിസ്റ്റം ക്രമീകരിക്കും.


## സെറ്റപ്പ്


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## പാറ്റേൺ 1: വ്യക്തമായ ഏജന്റ് നിർദ്ദേശങ്ങൾ

ഏറ്റവും വലിയ സ്വാധീനമുള്ള പാറ്റേണും ഏറ്റവും ലളിതവുമാണ്: നിങ്ങളുടെ ഏജന്റിന് വ്യക്തമായ, വിശദമായ നിർദ്ദേശങ്ങൾ എഴുതുക.

നല്ല നിർദ്ദേശങ്ങൾ നിർവചിക്കുന്നത്:
- ഏജന്റ് അഹങ്കാരമുള്ളവരാണെന്ന് (വ്യക്തിത്വവും ടോണും)
- അത് എന്ത് ചെയ്യാൻ ആഗ്രഹിക്കുന്നു (പടിയേറി ചുമതലകൾ)
- അതിന്റെ പെരുമാറ്റം എങ്ങനെ വേണമെന്ന് (പരിധികളും ശൈലിയും)

താഴെ, ഞങ്ങൾ യാത്രാ കൺസൈർജ് ഏജന്റ് സൃഷ്ടിക്കുന്നു, ഓരോ പ്രതികരണവും രൂപം നൽകുന്ന വ്യക്തമായ നിർദ്ദേശങ്ങളോടുകൂടി.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## പാറ്റേൺ 2: പൈഡാന്റിക് മോഡലുകൾ ഉപയോഗിച്ചുള്ള ഘടനയുള്ള ഔട്ട്പുട്ട്

സ്വതന്ത്ര-രൂപത്തിലുള്ള എഴുത്ത് സംഭാഷണത്തിനായി സഹായിക്കും, എങ്കിലും താഴെയുള്ള സിസ്റ്റങ്ങൾ ഘടനയുള്ള ഡേറ്റ ആവശ്യമാണ്.
**പൈഡാന്റിക് മോഡലുകൾ** ഒപ്പം ഒരു **ടൂൾ ഫംഗ്ഷൻ** ജോഡിച്ചാലാണ് നാം ചെയ്യാൻ പറ്റുന്നത്:

- ഏജന്റിന്റെ ഔട്ട്പുട്ടിന് കൃത്യമായ ഒരു സ്‌കീമ നിർവചിക്കുക
- മറുപടികളെ സ്വയം സാധൂക്കാര്യമാക്കുക
- ഏജന്റ് ഫലങ്ങളെ ആപ്ലിക്കേഷൻ ലാജിക്കിൽ വിശ്വാസയോഗ്യമായി സംയോജിപ്പിക്കുക

എന്ഫോഴ്സ്മെന്റിന്റെ കിഴിവ് ഏജണ്ട് റൺ ചെയ്യുമ്പോൾ `response_format` പാസ്സ് ചെയ്യാനാണ്. ഇത് മോഡലിനെ
സ്വതന്ത്ര എഴുത്തിന്റെ പകരം സാധൂകരിക്കപ്പെട്ട `TravelRecommendations` ഒബ്ജക്റ്റ് (ഉപലബ്ധം `response.value`ൽ)
നൽകാൻ നിർബന്ധിക്കുന്നു. `get_destination_details` ടൂൾ പോലും ടൈപ്പ്ഡായ
`DestinationRecommendation` തിരികെ നൽകുന്നു, അതിനാൽ ഡേറ്റ അവസാനത്തോളം ഘടനയുള്ളതാണ്.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## പാറ്റേൺ 3: ഏകദേശം ഉത്തരവാദിത്തം ഉളള ഏജന്റുകൾ

സങ്കീർണമായ ജോലികൾ വിവിധ കേന്ദ്രീകൃത ഏജന്റുകൾക്ക് വിഭജിക്കുന്നത് ഉപകാരപ്രദമാണ്, ഓരോ ഒരേ ഉത്തരവാദിത്തം ഉളളവരാകുന്നു:

- ഒരിടം വിദഗ്ധൻ (Destination Expert) ആണെന്ന് അറിയുന്നത് സ്ഥലങ്ങളും ലഭ്യതയും
- ഒരു ലജിസ്റ്റിക്സ് പ്ലാനർ (Logistics Planner) தான் വിമാനങ്ങളും ഹോട്ടലുകളും യാത്രാപഥങ്ങളും കൈകാര്യം ചെയ്യുന്നത്

ഇത് സോഫ്റ്റ്വെയർ എഞ്ചിനിയറിംഗ് സിദ്ധാന്തമായ *separation of concerns* എന്നതിനെ പ്രതിനിധീകരിക്കുന്നു — ഓരോ ഏജന്റും സ്വതന്ത്രമായും പരീക്ഷിക്കാനും പരിപാലിക്കാനും മെച്ചപ്പെടുത്താനും എളുപ്പമാണ്.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## സംക്ഷേപം

ഈ പാഠത്തിൽ, ഞങ്ങൾ ഒരു ട്രാവൽ ശുപാർശാ രംഗത്തിന് മൂന്ന് ഏജൻറിക് ഡിസൈൻ പാറ്റേണുകൾ പ്രയോഗിച്ചു:

| പാറ്റേൺ | പ്രധാന ആശയം | ഗുണം |
|---|---|---|
| **സ്വച്ഛമായ നിർദ്ദേശങ്ങൾ** | വ്യക്തിത്വം, ചുമതലകൾ, നിയന്ത്രണങ്ങൾ മുൻകൂട്ടി നിർവചിക്കുക | സ്ഥിരതയുള്ള, ബ്രാൻഡിനോട് ചേർന്ന് ഏജൻറ് പ്രവർത്തനം |
| **സംഘടിത ഔട്ട്പുട്ട്** | പ്രതികരണ രൂപമായി Pydantic മോഡലുകൾ ഉപയോഗിക്കുക | പരിശോധന കഴിഞ്ഞ, യന്ത്രം വായിക്കാൻ കഴിയുന്ന ഫലങ്ങൾ |
| **ഒറ്റ ചുമതല** | ഓരോ ഏജന്‍റിനും ഏകാഗ്രതയുള്ള ഒരു ജോലി നൽകുക | പരീക്ഷണവും പരിപാലനവും ലളിതമാക്കുന്നു |

ഈ പാറ്റേണുകൾ സ്വാഭാവികമായി ചേർത്ത് ഉപയോഗിക്കാമെന്ന് — നിങ്ങള്‍ക്ക് വ്യക്തമായി നിർദ്ദേശങ്ങൾ, ഘടനാപരമായ ഔട്ട്പുട്ട് എന്നിവ ഒറ്റ ചുമതലയുള്ള ഏജന്റിനുള്ളിൽ കൂട്ടിച്ചേർത്ത് സുസ്ഥിരവും ഉത്പാദനത്തിനൊരുങ്ങിയതുമായ സംവിധാനങ്ങൾ നിർമ്മിക്കാൻ കഴിയും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
